# GPU Pretrained-Model Fine-Tuning Benchmark

This notebook is the **Step 30 / pretrained-model benchmark path** for the Transaction Data Intelligence Framework.

It deliberately answers two separate questions:

1. **Representation question:** on the *same selected records and same split*, does E0 (raw), E1 (quality processed), or E2 (feature engineered) work best with a pretrained Transformer?
2. **Fine-tuning question:** for the same representation, how do **frozen-head**, **LoRA/PEFT**, and **full fine-tuning** compare?

The default checkpoint is `distilbert/distilbert-base-uncased` (67M parameters), which is small enough for a normal Colab GPU and is intended for downstream sequence classification. This notebook does **not** replace the built-in tabular Transformer. It is a separate pretrained-text benchmark.

### Experimental controls
- one `DataPreparer` instance selects the subset once;
- E0/E1/E2 reuse the exact same `_row_id` values;
- the split and seed are held constant;
- preprocessing level and fine-tuning method are varied explicitly;
- validation chooses the F1 threshold; test remains untouched until final evaluation;
- PR-AUC is the primary metric for imbalanced classification.

> The Step-11 built-in-model token IDs are **not** fed into DistilBERT. A pretrained language model must use its own tokenizer. We reuse the same rows/splits and render each level as text before applying the pretrained tokenizer.

In [ ]:
# Colab/Kaggle: run once after selecting a GPU runtime.
%pip install -q "transformers>=4.56" "peft>=0.17" "accelerate>=1.0" "scikit-learn>=1.3" "pyarrow>=14" "pyyaml>=6"

In [ ]:
from pathlib import Path
import gc, json, math, os, random, sys, time

import numpy as np
import pandas as pd
import torch
from sklearn import metrics as skm

# If the notebook is opened from this repository, this resolves automatically.
# In Colab, either upload/unzip the repository and %cd into it, or clone your repository first.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / "transaction-data-intelligence" / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT / "transaction-data-intelligence"
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the project root (the folder containing src/ and config.yaml).")
sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.roles import detect_roles, schema_for_profiling
from src.ingestion.schema_detector import detect_schema
from src.preprocessing.levels import DataPreparer, infer_roles
from src.representation.text_builder import spec_for
from src.evaluation.metrics import best_f1_threshold, classification_metrics
from src.utils.config import load_config

print("Project:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Experiment configuration

For the first smoke test, use a small subset (for example 2,000–6,000 rows), one seed, and 1–2 epochs. Once the notebook works end-to-end, increase the subset and run seeds `42, 123, 456`.

`TARGET`, `ENTITY`, and `DATETIME` may be left as `None` to use automatic detection. For a controlled benchmark on the main fraud dataset, setting them explicitly is safer.

In [ ]:
# ---- data ----
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "transactions.csv"  # change this
TARGET = None       # e.g. "is_fraud"
ENTITY = None       # e.g. "cc_num"; use "" to explicitly disable
DATETIME = None     # e.g. "trans_date_trans_time"; use "" to explicitly disable

SUBSET_ROWS = 6000  # exact total rows when enough labelled rows are available; use "full" for all
SEED = 42
LEVELS = ["E0", "E1", "E2"]

# ---- pretrained model ----
MODEL_ID = "distilbert/distilbert-base-uncased"
MAX_LENGTH = 256
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
EPOCHS = 3
PATIENCE = 2
WEIGHT_DECAY = 0.01

# Start with these two. Turn on full only after the smoke test succeeds.
METHODS = ["frozen_head", "lora"]  # options: frozen_head | lora | full

LEARNING_RATES = {
    "frozen_head": 5e-4,
    "lora": 2e-4,
    "full": 2e-5,
}

# DistilBERT attention projection layer names. Change only if MODEL_ID changes architecture.
LORA_TARGET_MODULES = ["q_lin", "v_lin"]
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

OUTPUT_DIR = PROJECT_ROOT / "experiments" / "pretrained_gpu"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load CSV/Parquet/JSON. Keep this simple and explicit for benchmark reproducibility.
def load_frame(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix in {".json", ".jsonl"}:
        return pd.read_json(path, lines=(suffix == ".jsonl"))
    raise ValueError(f"Unsupported input: {path}")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Upload/copy the dataset there or change DATA_PATH above."
    )

df = load_frame(DATA_PATH)
print(f"Loaded {len(df):,} rows x {df.shape[1]:,} columns")
df.head(3)

In [ ]:
# Detect schema/roles using the same project code as the Streamlit app.
cfg = load_config()
schema = detect_schema(df, dataset_id=DATA_PATH.stem)
detected = detect_roles(df, schema, target=TARGET)

roles = detected.with_overrides(
    df,
    target=TARGET if TARGET is not None else detected.target,
    entity=ENTITY if ENTITY is not None else detected.entity,
    datetime=DATETIME if DATETIME is not None else detected.datetime,
    task="binary_classification",
)
profile_schema = schema_for_profiling(schema, roles, df)
inferred = infer_roles(df, profile_schema, roles)

print("Target:", inferred.target)
print("Entity:", inferred.entity)
print("Datetime:", inferred.time)
print("Amount:", inferred.amount)
print("Category:", inferred.category)
print("Merchant:", inferred.merchant)

In [ ]:
# IMPORTANT: construct ONE preparer, select/split ONCE, then build all levels from it.
preparer = DataPreparer(df, profile_schema, inferred, cfg, rows=SUBSET_ROWS, seed=SEED)
split_info = preparer.prepare_split()
prepared = {level: preparer.build(level) for level in LEVELS}

# Prove controlled subset/split consistency before training anything.
for split in ["train", "validation", "test"]:
    ids = [prepared[level].frames[split]["_row_id"].tolist() for level in LEVELS]
    assert all(x == ids[0] for x in ids[1:]), f"Row mismatch in {split}"

print("Controlled subset verified across", LEVELS)
print(json.dumps(split_info["sample"], indent=2, default=str))

## 2. Convert each E-level to pretrained-model text

The representation is deliberately generic (`Tabular row:` + `feature=value` pairs). The target is never included in the text. E0/E1/E2 may have different features; that difference is the variable under study.

In [ ]:
def render_level_text(level_obj):
    spec = spec_for(level_obj, question="")
    spec.header = "Tabular row:"
    spec.question = ""
    out = {}
    for split, frame in level_obj.frames.items():
        text = spec.render(frame).str.rstrip()
        out[split] = pd.DataFrame({
            "_row_id": frame["_row_id"].values,
            "text": text.values,
            "label": frame["_target"].astype(int).values,
            "_weight": frame["_weight"].astype(float).values,
        })
    return out

text_data = {level: render_level_text(obj) for level, obj in prepared.items()}
for level in LEVELS:
    print(level, "fields:", len(prepared[level].features), "example:")
    print(text_data[level]["train"].iloc[0]["text"][:500])
    print()


## 3. Pretrained classifier + three fine-tuning modes

- **frozen_head**: pretrained encoder frozen; only the classification head trains.
- **lora**: pretrained base weights frozen; low-rank adapters train in selected attention projections, plus the classification head.
- **full**: every parameter trains. This costs the most GPU memory and generally uses a much smaller learning rate.

Hugging Face PEFT recommends LoRA as a common starting point for parameter-efficient fine-tuning. Keep `LORA_TARGET_MODULES` architecture-specific.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class TextDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length):
        self.frame = frame.reset_index(drop=True)
        self.enc = tokenizer(
            self.frame["text"].tolist(),
            truncation=True,
            max_length=max_length,
            padding=False,
        )
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        return {
            "input_ids": self.enc["input_ids"][i],
            "attention_mask": self.enc["attention_mask"][i],
            "labels": int(self.frame.loc[i, "label"]),
        }

class PadCollator:
    def __init__(self, tokenizer): self.tokenizer = tokenizer
    def __call__(self, batch):
        labels = torch.tensor([x.pop("labels") for x in batch], dtype=torch.long)
        padded = self.tokenizer.pad(batch, padding=True, return_tensors="pt")
        padded["labels"] = labels
        return padded


def build_model(method: str, model_id: str):
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
    if method == "full":
        return model

    if method == "frozen_head":
        for p in model.parameters():
            p.requires_grad = False
        trainable_markers = ("classifier", "pre_classifier", "score", "classification_head")
        for name, p in model.named_parameters():
            if any(m in name for m in trainable_markers):
                p.requires_grad = True
        if not any(p.requires_grad for p in model.parameters()):
            raise RuntimeError("Could not identify a classification head to unfreeze for this architecture")
        return model

    if method == "lora":
        from peft import LoraConfig, TaskType, get_peft_model
        lora_cfg = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGET_MODULES,
            bias="none",
        )
        return get_peft_model(model, lora_cfg)

    raise ValueError(method)


def parameter_counts(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable, 100 * trainable / max(total, 1)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("This benchmark is intended for a GPU runtime. In Colab: Runtime -> Change runtime type -> GPU.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
collator = PadCollator(tokenizer)
print("Device:", DEVICE, torch.cuda.get_device_name(0))

In [ ]:
@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    probs, labels = [], []
    for batch in loader:
        y = batch.pop("labels")
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch).logits
        probs.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(labels), np.concatenate(probs)


def make_loader(frame, shuffle=False, batch_size=BATCH_SIZE):
    ds = TextDataset(frame, tokenizer, MAX_LENGTH)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collator, num_workers=0)


def train_one(level: str, method: str, seed: int = SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    rows = text_data[level]
    train_loader = make_loader(rows["train"], shuffle=True, batch_size=BATCH_SIZE)
    val_loader = make_loader(rows["validation"], shuffle=False, batch_size=EVAL_BATCH_SIZE)
    test_loader = make_loader(rows["test"], shuffle=False, batch_size=EVAL_BATCH_SIZE)

    model = build_model(method, MODEL_ID).to(DEVICE)
    total, trainable, pct = parameter_counts(model)
    print(f"{level}/{method}: trainable {trainable:,}/{total:,} ({pct:.3f}%)")

    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATES[method], weight_decay=WEIGHT_DECAY,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    best = {"pr_auc": -1.0, "state": None, "epoch": None}
    stale = 0
    started = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(**batch)
                loss = out.loss
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss.detach().cpu()))

        yv, pv = predict_probs(model, val_loader)
        wv = rows["validation"]["_weight"].to_numpy()
        pr = skm.average_precision_score(yv, pv, sample_weight=wv) if len(np.unique(yv)) == 2 else float("nan")
        print(f"epoch={epoch} loss={np.mean(losses):.4f} weighted_val_PR_AUC={pr:.5f}")
        if pr > best["pr_auc"]:
            best = {
                "pr_auc": float(pr),
                "epoch": epoch,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            }
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print("early stopping")
                break

    model.load_state_dict(best["state"])
    yv, pv = predict_probs(model, val_loader)
    threshold = best_f1_threshold(yv, pv, rows["validation"]["_weight"].to_numpy())
    yt, pt = predict_probs(model, test_loader)
    metrics = classification_metrics(
        yt, pt, rows["test"]["_weight"].to_numpy(), threshold=threshold
    )
    result = {
        "level": level,
        "method": method,
        "model_id": MODEL_ID,
        "seed": seed,
        "subset_rows": SUBSET_ROWS,
        "max_length": MAX_LENGTH,
        "epochs_configured": EPOCHS,
        "best_epoch": best["epoch"],
        "learning_rate": LEARNING_RATES[method],
        "trainable_parameters": trainable,
        "total_parameters": total,
        "trainable_percent": pct,
        "threshold_from_validation": threshold,
        "validation_pr_auc": best["pr_auc"],
        "test": metrics,
        "seconds": time.time() - started,
    }
    del model, opt, scaler
    gc.collect(); torch.cuda.empty_cache()
    return result

## 4. Smoke test first

Start with one level and one method. If this succeeds, run the controlled matrix in the next cell.

In [ ]:
# Example smoke test:
# smoke = train_one("E0", "lora", SEED)
# print(json.dumps(smoke, indent=2, default=str))

## 5. Controlled E0/E1/E2 × fine-tuning-method matrix

This can take time. For an initial Colab T4 test, keep `METHODS=["frozen_head", "lora"]`, `EPOCHS=1-3`, and a small subset. Add `full` only after the pipeline is validated.

In [ ]:
results = []
for method in METHODS:
    for level in LEVELS:
        print("\n" + "=" * 90)
        result = train_one(level, method, SEED)
        results.append(result)
        out = OUTPUT_DIR / f"{level}_{method}_seed{SEED}.json"
        out.write_text(json.dumps(result, indent=2, default=str))
        print("saved", out)

summary_rows = []
for r in results:
    m = r["test"]
    summary_rows.append({
        "level": r["level"], "method": r["method"],
        "PR_AUC": m["pr_auc"], "ROC_AUC": m["roc_auc"],
        "precision": m["precision"], "recall": m["recall"], "F1": m["f1"],
        "best_epoch": r["best_epoch"], "trainable_%": r["trainable_percent"],
        "seconds": r["seconds"],
    })
summary = pd.DataFrame(summary_rows).sort_values(["method", "level"])
summary

## 6. Multi-seed benchmark

Only do this after the single-seed matrix works. The experiment is most useful when E0/E1/E2 share the exact rows for every seed. Rebuilding a `DataPreparer` per seed changes the subset, so for the strictest representation comparison keep the subset seed fixed and vary only the **model seed** first. Then, as a separate robustness experiment, vary the sampling seed too.

In [ ]:
# Recommended final model-seed comparison on one fixed data subset:
# MODEL_SEEDS = [42, 123, 456]
# multi = []
# for model_seed in MODEL_SEEDS:
#     for method in METHODS:
#         for level in LEVELS:
#             multi.append(train_one(level, method, model_seed))
# (OUTPUT_DIR / "combined_results.json").write_text(json.dumps(multi, indent=2, default=str))

## Interpretation

Do **not** conclude that preprocessing is better or worse from a single seed or tiny subset.

If E0 still wins with LoRA/full fine-tuning, investigate representation loss and E2 feature noise next. If E1/E2 improve relative to E0 only after fine-tuning the pretrained encoder, that suggests the richer representations needed more adaptable model capacity than the frozen baseline supplied.

For the product report, preserve:
- model checkpoint and revision if pinned;
- preprocessing level;
- selected row hash/split metadata;
- model seed;
- method (frozen/LoRA/full);
- LoRA configuration;
- best validation epoch and threshold;
- weighted test PR-AUC, ROC-AUC, precision, recall, F1;
- GPU name and runtime.